# Model Building v2 (Dual-Method, Anti-Leakage)

本 notebook 依照 model bulilding.md 實作：
1. 方法 A（可解釋特徵）
2. 方法 B（語義特徵 + 可解釋特徵）
3. 所有特徵選擇放入 Pipeline（每 fold 獨立）
4. 使用 Stratified 10-Fold 比較，最後選最佳模型產出 submission

In [3]:
import re
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, classification_report

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print('HAS_XGB:', HAS_XGB)
print('HAS_LGB:', HAS_LGB)

HAS_XGB: True
HAS_LGB: True


## Data Loading

In [4]:
DATA_DIR = Path('./data')
TRAIN_PATH = DATA_DIR / 'boy or girl 2025 train_missingValue.csv'
TEST_PATH = DATA_DIR / 'boy or girl 2025 test no ans_missingValue.csv'

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(f'Missing data files:\n{TRAIN_PATH}\n{TEST_PATH}')

df_train_full = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

df_train, df_holdout = train_test_split(
    df_train_full,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=df_train_full['gender']
 )

print('Train full shape:', df_train_full.shape)
print('Train 80% shape:', df_train.shape)
print('Holdout 20% shape:', df_holdout.shape)
print('Test shape:', df_test.shape)
print('Train full gender distribution:', df_train_full['gender'].value_counts().to_dict())
print('Train 80% gender distribution:', df_train['gender'].value_counts().to_dict())
print('Holdout 20% gender distribution:', df_holdout['gender'].value_counts().to_dict())

Train full shape: (423, 11)
Train 80% shape: (338, 11)
Holdout 20% shape: (85, 11)
Test shape: (426, 11)
Train full gender distribution: {1: 316, 2: 107}
Train 80% gender distribution: {1: 253, 2: 85}
Holdout 20% gender distribution: {1: 63, 2: 22}


In [34]:
STAR_ELEMENT_MAP = {
    'aries': 'fire', 'leo': 'fire', 'sagittarius': 'fire',
    'taurus': 'earth', 'virgo': 'earth', 'capricorn': 'earth',
    'gemini': 'air', 'libra': 'air', 'aquarius': 'air',
    'cancer': 'water', 'scorpio': 'water', 'pisces': 'water',
    # Chinese aliases
    '牡羊座': 'fire', '獅子座': 'fire', '射手座': 'fire',
    '金牛座': 'earth', '處女座': 'earth', '摩羯座': 'earth',
    '雙子座': 'air', '天秤座': 'air', '水瓶座': 'air',
    '巨蟹座': 'water', '天蠍座': 'water', '雙魚座': 'water'
}

MALE_WORDS = ['handsome', 'basketball', 'loser', 'game', 'esports']
FEMALE_WORDS = ['beautiful', 'cute', 'makeup', 'skincare', 'perfume']

class FeatureBuilder(BaseEstimator, TransformerMixin):
    """更嚴謹的特徵構建器：先過濾異常值再計算統計量，確保 BMI 不因異常值失效。"""

    def __init__(self, method='A', n_semantic=20):
        self.method = method
        self.n_semantic = n_semantic
        self.vectorizer_ = None
        self.svd_ = None
        self.height_bin_edges_ = None
        # 新增：儲存訓練集的身高體重中位數，用於計算 BMI 前的簡易補值
        self.height_median_ = 165.0
        self.weight_median_ = 60.0

    def _to_df(self, X):
        if isinstance(X, pd.DataFrame):
            return X.copy()
        return pd.DataFrame(X)

    def _clean_outliers(self, df):
        """內部使用的異常值清洗邏輯"""
        out = df.copy()
        if 'height' in out.columns:
            out.loc[(out['height'] < 140) | (out['height'] > 200), 'height'] = np.nan
        if 'weight' in out.columns:
            out.loc[(out['weight'] < 35) | (out['weight'] > 120), 'weight'] = np.nan
        return out

    def fit(self, X, y=None):
        X = self._to_df(X)
        # 1. 先清洗異常值，避免分位數被拉歪
        X_cleaned = self._clean_outliers(X)
        
        # 2. 計算身高分箱邊界
        height = pd.to_numeric(X_cleaned.get('height', pd.Series(dtype=float)), errors='coerce')
        self.height_median_ = height.median() if not height.isna().all() else 165.0
        
        h1, h2 = height.quantile(0.33), height.quantile(0.67)
        if pd.isna(h1) or pd.isna(h2) or h1 >= h2:
            h1, h2 = 160.0, 175.0
        self.height_bin_edges_ = (-np.inf, h1, h2, np.inf)

        # 3. 儲存體重中位數
        weight = pd.to_numeric(X_cleaned.get('weight', pd.Series(dtype=float)), errors='coerce')
        self.weight_median_ = weight.median() if not weight.isna().all() else 60.0

        # 4. 方法 B 的語義特徵
        if self.method == 'B':
            corpus = X.get('self_intro', pd.Series('', index=X.index)).fillna('').astype(str)
            self.vectorizer_ = TfidfVectorizer(max_features=300, ngram_range=(1, 2), min_df=2)
            tfidf = self.vectorizer_.fit_transform(corpus)
            comp = min(self.n_semantic, max(2, tfidf.shape[1] - 1)) if tfidf.shape[1] > 2 else 2
            self.svd_ = TruncatedSVD(n_components=comp, random_state=RANDOM_SEED)
            self.svd_.fit(tfidf)

        return self

    def transform(self, X):
        X = self._to_df(X)
        # 1. 清洗異常值
        out = self._clean_outliers(X)

        # Explicitly remove excluded raw features from the feature frame.
        drop_cols = [c for c in ['yt', 'iq', 'phone_os', 'sleepiness'] if c in out.columns]
        if drop_cols:
            out = out.drop(columns=drop_cols)

        for c in ['height', 'weight', 'fb_friends'] :
            if c in out.columns:
                out[c] = pd.to_numeric(out[c], errors='coerce')

        # 2. 安全計算 BMI
        # 使用填補後的身高體重來算 BMI，確保 BMI 欄位不會有太多 NaN 影響後續 Selector
        temp_h = out['height'].fillna(self.height_median_) / 100.0
        temp_w = out['weight'].fillna(self.weight_median_)
        out['bmi'] = temp_w / (temp_h ** 2)
        
        out['is_tall'] = (out['height'] >= 170).astype(int)
        out['height_pct'] = out['height'].rank(pct=True)

        edges = self.height_bin_edges_ if self.height_bin_edges_ is not None else (-np.inf, 160, 175, np.inf)
        out['height_bin'] = pd.cut(out['height'], bins=edges, labels=['low', 'mid', 'high'])

        # 3. 其他特徵（類別、文字等）保持不變...
        star = out.get('star_sign', pd.Series('', index=out.index)).fillna('Unknown').astype(str).str.strip().str.lower()
        out['star_element'] = star.map(STAR_ELEMENT_MAP).fillna('unknown')

        intro = out.get('self_intro', pd.Series('', index=out.index)).fillna('').astype(str)
        out['has_intro'] = (intro.str.strip() != '').astype(int)
        out['intro_len'] = intro.str.len().astype(float)
        out['uses_emoji'] = intro.str.contains(r'[\U0001F300-\U0001FAFF❤️]').astype(int)
        out['tilde_count'] = intro.str.count(r'[~～]').astype(float)
        out['exclamation_count'] = intro.str.count(r'!').astype(float)

        intro_lc = intro.str.lower()
        male_hits = intro_lc.apply(lambda s: sum(int(w in s) for w in MALE_WORDS))
        female_hits = intro_lc.apply(lambda s: sum(int(w in s) for w in FEMALE_WORDS))
        out['keyword_score'] = (male_hits - female_hits).astype(float)

        if self.method == 'B' and self.vectorizer_ is not None and self.svd_ is not None:
            tfidf = self.vectorizer_.transform(intro)
            sem = self.svd_.transform(tfidf)
            for i in range(sem.shape[1]):
                out[f'sem_{i}'] = sem[:, i]

        return out

In [35]:
def build_preprocessor(feature_cols):
    numeric_cols = [
        'height', 'weight', 'fb_friends', 'bmi', 'is_tall',
        'height_pct', 'has_intro', 'intro_len', 'uses_emoji', 'tilde_count',
        'exclamation_count', 'keyword_score'
    ]
    numeric_cols += [c for c in feature_cols if c.startswith('sem_')]

    # Excluded from modeling: yt, iq, phone_os, sleepiness.
    categorical_cols = ['star_element', 'height_bin']

    numeric_cols = [c for c in numeric_cols if c in feature_cols]
    categorical_cols = [c for c in categorical_cols if c in feature_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median'), numeric_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
                ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
            ]), categorical_cols)
        ],
        remainder='drop'
    )
    return preprocessor


def make_method_pipeline(method='A', model_name='xgb', selector_name='l1', scale_pos_weight=1.0):
    feat = FeatureBuilder(method=method, n_semantic=20)

    # Build a temporary transformed frame to determine available columns.
    temp = feat.fit_transform(df_train.drop(columns=['gender']))
    feature_cols = temp.columns.tolist()
    preprocessor = build_preprocessor(feature_cols)

    selector_l1 = SelectFromModel(
        LogisticRegression(
            penalty='l1', solver='liblinear', class_weight='balanced',
            C=0.5, random_state=RANDOM_SEED, max_iter=3000
        )
    )

    selector_rfe = RFE(
        estimator=LogisticRegression(
            penalty='l2', solver='liblinear', class_weight='balanced',
            random_state=RANDOM_SEED, max_iter=3000
        ),
        n_features_to_select=20,
        step=0.2
    )

    if model_name == 'xgb':
        if not HAS_XGB:
            raise RuntimeError('xgboost is not installed')
        clf = xgb.XGBClassifier(
            n_estimators=320,
            learning_rate=0.04,
            max_depth=5,
            min_child_weight=6,
            reg_alpha=1.5,
            reg_lambda=5.0,
            subsample=0.8,
            colsample_bytree=0.8,
            objective='binary:logistic',
            eval_metric='auc',
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_SEED
        )
    elif model_name == 'lgb':
        if not HAS_LGB:
            raise RuntimeError('lightgbm is not installed')
        clf = lgb.LGBMClassifier(
            n_estimators=400,
            learning_rate=0.04,
            max_depth=5,
            num_leaves=31,
            min_child_samples=20,
            reg_alpha=1.0,
            reg_lambda=5.0,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight='balanced',
            random_state=RANDOM_SEED,
            verbose=-1
        )
    elif model_name == 'rf':
        clf = RandomForestClassifier(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=4,
            class_weight='balanced',
            random_state=RANDOM_SEED,
            n_jobs=-1
        )
    else:
        raise ValueError(f'Unsupported model_name: {model_name}')

    if selector_name == 'l1':
        selector = selector_l1
    elif selector_name == 'rfe':
        selector = selector_rfe
    else:
        raise ValueError(f'Unsupported selector_name: {selector_name}')

    pipe = Pipeline([
        ('feature', feat),
        ('preprocess', preprocessor),
        ('selector', selector),
        ('model', clf)
    ])
    return pipe

In [36]:
X_raw = df_train.drop(columns=['gender'])
y = df_train['gender'].astype(int)
y_bin = (y == 2).astype(int)

X_holdout_raw = df_holdout.drop(columns=['gender'])
y_holdout = df_holdout['gender'].astype(int)
y_holdout_bin = (y_holdout == 2).astype(int)

n_neg = int((y_bin == 0).sum())
n_pos = int((y_bin == 1).sum())
scale_pos_weight = n_neg / n_pos if n_pos else 1.0

print('scale_pos_weight (from train 80%):', round(scale_pos_weight, 4))

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)
scoring = {'roc_auc': 'roc_auc', 'f1_macro': 'f1_macro', 'accuracy': 'accuracy'}

candidates = []

# Method A: XGB (L1 + RFE)
if HAS_XGB:
    candidates.append(('A_xgb_l1', make_method_pipeline('A', 'xgb', selector_name='l1', scale_pos_weight=scale_pos_weight), 'A'))
    candidates.append(('A_xgb_rfe', make_method_pipeline('A', 'xgb', selector_name='rfe', scale_pos_weight=scale_pos_weight), 'A'))

# Method A: LGB (L1 + RFE)
if HAS_LGB:
    candidates.append(('A_lgb_l1', make_method_pipeline('A', 'lgb', selector_name='l1', scale_pos_weight=scale_pos_weight), 'A'))
    candidates.append(('A_lgb_rfe', make_method_pipeline('A', 'lgb', selector_name='rfe', scale_pos_weight=scale_pos_weight), 'A'))

# Method A: RF (L1 + RFE)
candidates.append(('A_rf_l1', make_method_pipeline('A', 'rf', selector_name='l1', scale_pos_weight=scale_pos_weight), 'A'))
candidates.append(('A_rf_rfe', make_method_pipeline('A', 'rf', selector_name='rfe', scale_pos_weight=scale_pos_weight), 'A'))

# Method B: LGB (L1 + RFE)
if HAS_LGB:
    candidates.append(('B_lgb_l1', make_method_pipeline('B', 'lgb', selector_name='l1', scale_pos_weight=scale_pos_weight), 'B'))
    candidates.append(('B_lgb_rfe', make_method_pipeline('B', 'lgb', selector_name='rfe', scale_pos_weight=scale_pos_weight), 'B'))

# Method B: XGB (L1 + RFE)
if HAS_XGB:
    candidates.append(('B_xgb_l1', make_method_pipeline('B', 'xgb', selector_name='l1', scale_pos_weight=scale_pos_weight), 'B'))
    candidates.append(('B_xgb_rfe', make_method_pipeline('B', 'xgb', selector_name='rfe', scale_pos_weight=scale_pos_weight), 'B'))

# Method B: RF (L1 + RFE)
candidates.append(('B_rf_l1', make_method_pipeline('B', 'rf', selector_name='l1', scale_pos_weight=scale_pos_weight), 'B'))
candidates.append(('B_rf_rfe', make_method_pipeline('B', 'rf', selector_name='rfe', scale_pos_weight=scale_pos_weight), 'B'))

results = []
for name, pipe, method_tag in candidates:
    cv = cross_validate(pipe, X_raw, y_bin, cv=skf, scoring=scoring, n_jobs=1, return_train_score=False)
    row = {
        'name': name,
        'method': method_tag,
        'roc_auc_mean': float(np.mean(cv['test_roc_auc'])),
        'roc_auc_std': float(np.std(cv['test_roc_auc'])),
        'f1_macro_mean': float(np.mean(cv['test_f1_macro'])),
        'f1_macro_std': float(np.std(cv['test_f1_macro'])),
        'accuracy_mean': float(np.mean(cv['test_accuracy'])),
        'accuracy_std': float(np.std(cv['test_accuracy']))
    }
    results.append(row)
    print(f"{name}: ROC-AUC={row['roc_auc_mean']:.4f}±{row['roc_auc_std']:.4f}, F1={row['f1_macro_mean']:.4f}±{row['f1_macro_std']:.4f}")

# Score-first selection with no method bias (F1 priority).
# Priority order:
# 1) higher F1-macro mean
# 2) higher ROC-AUC mean
# 3) lower F1-macro std (more stable on target metric)
# 4) higher accuracy mean
# 5) lower ROC-AUC std
# 6) lower accuracy std
results_df = pd.DataFrame(results).sort_values(
    ['f1_macro_mean', 'roc_auc_mean', 'f1_macro_std', 'accuracy_mean', 'roc_auc_std', 'accuracy_std'],
    ascending=[False, False, True, False, True, True]
).reset_index(drop=True)

print('\n=== Candidate Ranking (CV on train 80%) ===')
display(results_df)

best = results_df.iloc[0].copy()
best_name = best['name']
print('\nBest candidate selected (F1-first, no method bias):', best_name)

scale_pos_weight (from train 80%): 2.9765
A_xgb_l1: ROC-AUC=0.9013±0.0468, F1=0.8337±0.0624
A_xgb_rfe: ROC-AUC=0.9127±0.0475, F1=0.8390±0.0511
A_lgb_l1: ROC-AUC=0.9012±0.0480, F1=0.8368±0.0634
A_lgb_rfe: ROC-AUC=0.9083±0.0512, F1=0.8393±0.0580
A_rf_l1: ROC-AUC=0.9011±0.0488, F1=0.8317±0.0516
A_rf_rfe: ROC-AUC=0.9084±0.0518, F1=0.8226±0.0592
B_lgb_l1: ROC-AUC=0.9041±0.0491, F1=0.8442±0.0570
B_lgb_rfe: ROC-AUC=0.8961±0.0629, F1=0.8138±0.0827
B_xgb_l1: ROC-AUC=0.9006±0.0451, F1=0.8278±0.0663
B_xgb_rfe: ROC-AUC=0.8940±0.0597, F1=0.8046±0.0746
B_rf_l1: ROC-AUC=0.9033±0.0498, F1=0.8360±0.0633
B_rf_rfe: ROC-AUC=0.8967±0.0633, F1=0.8131±0.0832

=== Candidate Ranking (CV on train 80%) ===


,name,method,roc_auc_mean,roc_auc_std,f1_macro_mean,f1_macro_std,accuracy_mean,accuracy_std
0,B_lgb_l1,B,0.904141,0.049120,0.844207,0.056979,0.878699,0.048327
1,A_lgb_rfe,A,0.908250,0.051151,0.839337,0.057993,0.872816,0.049627
2,A_xgb_rfe,A,0.912675,0.047509,0.838992,0.051114,0.869786,0.042379
3,A_lgb_l1,A,0.901162,0.047976,0.836761,0.063397,0.872906,0.052659
4,B_rf_l1,B,0.903346,0.049792,0.836045,0.063276,0.869875,0.049846
5,A_xgb_l1,A,0.901274,0.046797,0.833652,0.062396,0.866934,0.052945
6,A_rf_l1,A,0.901124,0.048845,0.831656,0.051612,0.866845,0.040334
7,B_xgb_l1,B,0.900628,0.045090,0.827799,0.066271,0.861052,0.057264
8,A_rf_rfe,A,0.908417,0.051774,0.822601,0.059233,0.860963,0.049476
9,B_lgb_rfe,B,0.896132,0.062891,0.813844,0.082681,0.851872,0.074841



Best candidate selected (F1-first, no method bias): B_lgb_l1


In [37]:
# Optional: soft voting between best A and best B candidates
method_best = {}
for method_tag in ['A', 'B']:
    sub = results_df[results_df['method'] == method_tag]
    if not sub.empty:
        method_best[method_tag] = sub.iloc[0]['name']

name_to_pipe = {name: pipe for name, pipe, _ in candidates}

if 'A' in method_best and 'B' in method_best:
    a_pipe = name_to_pipe[method_best['A']]
    b_pipe = name_to_pipe[method_best['B']]
    voter = VotingClassifier(
        estimators=[('A_best', a_pipe), ('B_best', b_pipe)],
        voting='soft'
    )
    vote_cv = cross_validate(voter, X_raw, y_bin, cv=skf, scoring=scoring, n_jobs=1)
    vote_row = {
        'name': 'Voting_A_B',
        'method': 'Ensemble',
        'roc_auc_mean': float(np.mean(vote_cv['test_roc_auc'])),
        'roc_auc_std': float(np.std(vote_cv['test_roc_auc'])),
        'f1_macro_mean': float(np.mean(vote_cv['test_f1_macro'])),
        'f1_macro_std': float(np.std(vote_cv['test_f1_macro'])),
        'accuracy_mean': float(np.mean(vote_cv['test_accuracy'])),
        'accuracy_std': float(np.std(vote_cv['test_accuracy']))
    }
    display(pd.DataFrame([vote_row]))

    # Adopt voting only when both mean and std are better than current best
    if (vote_row['roc_auc_mean'] > float(best['roc_auc_mean'])) and (vote_row['roc_auc_std'] <= float(best['roc_auc_std'])):
        best_name = 'Voting_A_B'
        name_to_pipe['Voting_A_B'] = voter
        print('Voting model adopted as final best model.')
    else:
        print('Voting model not adopted (did not beat best model on mean+std rule).')

# Final test on holdout 20%
final_model = name_to_pipe[best_name]
final_model.fit(X_raw, y_bin)
holdout_proba = final_model.predict_proba(X_holdout_raw)[:, 1]
holdout_pred_bin = (holdout_proba >= 0.5).astype(int)
holdout_pred = holdout_pred_bin + 1

holdout_auc = roc_auc_score(y_holdout_bin, holdout_proba)
holdout_f1 = f1_score(y_holdout, holdout_pred, average='macro')
holdout_acc = accuracy_score(y_holdout, holdout_pred)

print('\n=== Final Holdout (20%) Test ===')
print('Best model used:', best_name)
print(f'Holdout ROC-AUC: {holdout_auc:.4f}')
print(f'Holdout F1-macro: {holdout_f1:.4f}')
print(f'Holdout Accuracy: {holdout_acc:.4f}')
print('\nClassification Report (holdout):')
print(classification_report(y_holdout, holdout_pred, digits=4))

holdout_result = pd.DataFrame({
    'id': df_holdout['id'].values,
    'gender_true': y_holdout.values,
    'gender_pred': holdout_pred,
    'proba_female': holdout_proba
})
holdout_result.to_csv('holdout_20_result.csv', index=False)
print('Saved: holdout_20_result.csv')

,name,method,roc_auc_mean,roc_auc_std,f1_macro_mean,f1_macro_std,accuracy_mean,accuracy_std
0,Voting_A_B,Ensemble,0.908637,0.050295,0.848971,0.051714,0.881818,0.045348


Voting model not adopted (did not beat best model on mean+std rule).

=== Final Holdout (20%) Test ===
Best model used: B_lgb_l1
Holdout ROC-AUC: 0.9423
Holdout F1-macro: 0.8867
Holdout Accuracy: 0.9059

Classification Report (holdout):
              precision    recall  f1-score   support

           1     0.9825    0.8889    0.9333        63
           2     0.7500    0.9545    0.8400        22

    accuracy                         0.9059        85
   macro avg     0.8662    0.9217    0.8867        85
weighted avg     0.9223    0.9059    0.9092        85

Saved: holdout_20_result.csv


In [39]:
# Inspect which features are selected by L1 SelectFromModel in A_rf_l1
probe_pipe = make_method_pipeline('A', 'rf', scale_pos_weight=scale_pos_weight)
probe_pipe.fit(X_raw, y_bin)

feature_step = probe_pipe.named_steps['feature']
preprocess_step = probe_pipe.named_steps['preprocess']
selector_step = probe_pipe.named_steps['selector']

X_feat = feature_step.transform(X_raw)
X_pre = preprocess_step.transform(X_feat)
pre_names = preprocess_step.get_feature_names_out()
selected_mask = selector_step.get_support()
selected_features = [name for name, keep in zip(pre_names, selected_mask) if keep]

print('Total features after preprocess:', len(pre_names))
print('Selected by L1:', int(selected_mask.sum()))
print('\nSelected feature names:')
for i, f in enumerate(selected_features, 1):
    print(f'{i:02d}. {f}')

Total features after preprocess: 21
Selected by L1: 9

Selected feature names:
01. num__height
02. num__weight
03. num__bmi
04. num__is_tall
05. num__height_pct
06. num__intro_len
07. num__keyword_score
08. cat__star_element_earth
09. cat__height_bin_mid


In [17]:
# Inspect selected features for multiple models (L1/RFE, A/B)
if 'candidates' not in globals():
    raise RuntimeError('candidates is not available. Run the model comparison cell first.')

name_to_pipe_local = {name: pipe for name, pipe, _ in candidates}

# Edit this list to inspect any models you want.
target_models = [
    'A_rf_l1',
    'B_rf_l1',
    'A_xgb_l1',
    'A_xgb_rfe'
]

available_models = [m for m in target_models if m in name_to_pipe_local]
missing_models = [m for m in target_models if m not in name_to_pipe_local]

if missing_models:
    print('Missing models (not in current candidates):', missing_models)

if not available_models:
    raise RuntimeError('No target models found in candidates.')


def inspect_selected_features(model_name):
    pipe = name_to_pipe_local[model_name]
    pipe.fit(X_raw, y_bin)

    feature_step = pipe.named_steps['feature']
    preprocess_step = pipe.named_steps['preprocess']
    selector_step = pipe.named_steps['selector']

    X_feat = feature_step.transform(X_raw)
    _ = preprocess_step.transform(X_feat)

    pre_names = preprocess_step.get_feature_names_out()
    selected_mask = selector_step.get_support()
    selected = [name for name, keep in zip(pre_names, selected_mask) if keep]

    sem_selected = [f for f in selected if f.startswith('num__sem_')]

    print(f'\n=== {model_name} ===')
    print('Total features after preprocess:', len(pre_names))
    print('Selected features:', len(selected))
    print('Selected semantic features (num__sem_*):', len(sem_selected))

    for i, feat_name in enumerate(selected, 1):
        print(f'{i:02d}. {feat_name}')

    return set(selected), selected, len(pre_names), len(sem_selected)


selected_sets = {}
summary_rows = []

for model_name in available_models:
    feature_set, selected_list, total_pre, sem_cnt = inspect_selected_features(model_name)
    selected_sets[model_name] = feature_set
    summary_rows.append({
        'model': model_name,
        'total_preprocessed_features': total_pre,
        'selected_features': len(selected_list),
        'selected_semantic_features': sem_cnt
    })

summary_df = pd.DataFrame(summary_rows).sort_values('model').reset_index(drop=True)
print('\n=== Summary ===')
display(summary_df)

# Pairwise comparison
print('\n=== Pairwise Intersections / Differences ===')
for i in range(len(available_models)):
    for j in range(i + 1, len(available_models)):
        m1, m2 = available_models[i], available_models[j]
        common = sorted(selected_sets[m1] & selected_sets[m2])
        only_m1 = sorted(selected_sets[m1] - selected_sets[m2])
        only_m2 = sorted(selected_sets[m2] - selected_sets[m1])

        print(f'\n[{m1}] vs [{m2}]')
        print('Common:', len(common))
        print('Only in', m1 + ':', len(only_m1))
        print('Only in', m2 + ':', len(only_m2))

        print('Sample common features:', common[:10])
        print('Sample only in', m1 + ':', only_m1[:10])
        print('Sample only in', m2 + ':', only_m2[:10])


=== A_rf_l1 ===
Total features after preprocess: 29
Selected features: 14
Selected semantic features (num__sem_*): 0
01. num__height
02. num__weight
03. num__sleepiness
04. num__iq
05. num__fb_friends
06. num__yt
07. num__bmi
08. num__intro_len
09. num__keyword_score
10. cat__star_element_earth
11. cat__phone_os_Android
12. cat__phone_os_Unknown
13. cat__height_bin_Unknown
14. cat__height_bin_mid

=== B_rf_l1 ===
Total features after preprocess: 49
Selected features: 14
Selected semantic features (num__sem_*): 0
01. num__height
02. num__weight
03. num__sleepiness
04. num__iq
05. num__fb_friends
06. num__yt
07. num__bmi
08. num__intro_len
09. num__keyword_score
10. cat__star_element_earth
11. cat__phone_os_Android
12. cat__phone_os_Unknown
13. cat__height_bin_Unknown
14. cat__height_bin_mid

=== A_xgb_l1 ===
Total features after preprocess: 29
Selected features: 14
Selected semantic features (num__sem_*): 0
01. num__height
02. num__weight
03. num__sleepiness
04. num__iq
05. num__fb_fri

,model,total_preprocessed_features,selected_features,selected_semantic_features
0,A_rf_l1,29,14,0
1,A_xgb_l1,29,14,0
2,A_xgb_rfe,29,20,0
3,B_rf_l1,49,14,0



=== Pairwise Intersections / Differences ===

[A_rf_l1] vs [B_rf_l1]
Common: 14
Only in A_rf_l1: 0
Only in B_rf_l1: 0
Sample common features: ['cat__height_bin_Unknown', 'cat__height_bin_mid', 'cat__phone_os_Android', 'cat__phone_os_Unknown', 'cat__star_element_earth', 'num__bmi', 'num__fb_friends', 'num__height', 'num__intro_len', 'num__iq']
Sample only in A_rf_l1: []
Sample only in B_rf_l1: []

[A_rf_l1] vs [A_xgb_l1]
Common: 14
Only in A_rf_l1: 0
Only in A_xgb_l1: 0
Sample common features: ['cat__height_bin_Unknown', 'cat__height_bin_mid', 'cat__phone_os_Android', 'cat__phone_os_Unknown', 'cat__star_element_earth', 'num__bmi', 'num__fb_friends', 'num__height', 'num__intro_len', 'num__iq']
Sample only in A_rf_l1: []
Sample only in A_xgb_l1: []

[A_rf_l1] vs [A_xgb_rfe]
Common: 10
Only in A_rf_l1: 4
Only in A_xgb_rfe: 10
Sample common features: ['cat__height_bin_Unknown', 'cat__height_bin_mid', 'cat__phone_os_Android', 'cat__phone_os_Unknown', 'cat__star_element_earth', 'num__bmi', '

In [18]:
# Sensitivity test: does larger L1 C keep semantic features for RF?
if 'X_raw' not in globals() or 'y_bin' not in globals():
    raise RuntimeError('Run the model comparison cell first so X_raw / y_bin are available.')


def make_rf_l1_with_c(method='B', c_value=0.5, scale_pos_weight=1.0):
    feat = FeatureBuilder(method=method, n_semantic=20)

    # Determine dynamic columns exactly like the main pipeline.
    temp = feat.fit_transform(df_train.drop(columns=['gender']))
    feature_cols = temp.columns.tolist()
    preprocessor = build_preprocessor(feature_cols)

    selector_l1_custom = SelectFromModel(
        LogisticRegression(
            penalty='l1', solver='liblinear', class_weight='balanced',
            C=c_value, random_state=RANDOM_SEED, max_iter=3000
        )
    )

    clf = RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=4,
        class_weight='balanced',
        random_state=RANDOM_SEED,
        n_jobs=-1
    )

    return Pipeline([
        ('feature', feat),
        ('preprocess', preprocessor),
        ('selector', selector_l1_custom),
        ('model', clf)
    ])


c_grid = [0.2, 0.5, 1.0, 2.0, 5.0]
skf_local = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)
scoring_local = {'roc_auc': 'roc_auc', 'f1_macro': 'f1_macro', 'accuracy': 'accuracy'}

rows = []

for c_val in c_grid:
    # Focus on Method B first (your concern: semantic features usefulness)
    pipe_b = make_rf_l1_with_c(method='B', c_value=c_val, scale_pos_weight=scale_pos_weight)
    pipe_b.fit(X_raw, y_bin)

    pre_names_b = pipe_b.named_steps['preprocess'].get_feature_names_out()
    mask_b = pipe_b.named_steps['selector'].get_support()
    selected_b = [name for name, keep in zip(pre_names_b, mask_b) if keep]
    sem_selected_b = [f for f in selected_b if f.startswith('num__sem_')]

    cv_b = cross_validate(pipe_b, X_raw, y_bin, cv=skf_local, scoring=scoring_local, n_jobs=1, return_train_score=False)

    rows.append({
        'method': 'B_rf_l1',
        'C': c_val,
        'selected_total': len(selected_b),
        'selected_sem': len(sem_selected_b),
        'roc_auc_mean': float(np.mean(cv_b['test_roc_auc'])),
        'f1_macro_mean': float(np.mean(cv_b['test_f1_macro'])),
        'accuracy_mean': float(np.mean(cv_b['test_accuracy']))
    })

    # Add A as reference baseline under same C.
    pipe_a = make_rf_l1_with_c(method='A', c_value=c_val, scale_pos_weight=scale_pos_weight)
    pipe_a.fit(X_raw, y_bin)

    pre_names_a = pipe_a.named_steps['preprocess'].get_feature_names_out()
    mask_a = pipe_a.named_steps['selector'].get_support()
    selected_a = [name for name, keep in zip(pre_names_a, mask_a) if keep]

    cv_a = cross_validate(pipe_a, X_raw, y_bin, cv=skf_local, scoring=scoring_local, n_jobs=1, return_train_score=False)

    rows.append({
        'method': 'A_rf_l1',
        'C': c_val,
        'selected_total': len(selected_a),
        'selected_sem': 0,
        'roc_auc_mean': float(np.mean(cv_a['test_roc_auc'])),
        'f1_macro_mean': float(np.mean(cv_a['test_f1_macro'])),
        'accuracy_mean': float(np.mean(cv_a['test_accuracy']))
    })

result_c = pd.DataFrame(rows).sort_values(['method', 'C']).reset_index(drop=True)

print('=== L1 C sensitivity (RF) ===')
display(result_c)

print('\n=== B_rf_l1 only (check semantic feature usage) ===')
display(result_c[result_c['method'] == 'B_rf_l1'])

=== L1 C sensitivity (RF) ===


,method,C,selected_total,selected_sem,roc_auc_mean,f1_macro_mean,accuracy_mean
0,A_rf_l1,0.2,11,0,0.899517,0.823395,0.864082
1,A_rf_l1,0.5,14,0,0.906942,0.815947,0.861052
2,A_rf_l1,1.0,18,0,0.906096,0.812300,0.852228
3,A_rf_l1,2.0,21,0,0.901154,0.813043,0.852050
4,A_rf_l1,5.0,21,0,0.905265,0.821719,0.857932
5,B_rf_l1,0.2,11,0,0.897536,0.823395,0.864082
6,B_rf_l1,0.5,14,0,0.906942,0.815947,0.861052
7,B_rf_l1,1.0,20,1,0.902799,0.825560,0.863993
8,B_rf_l1,2.0,29,7,0.894637,0.809184,0.852050
9,B_rf_l1,5.0,37,14,0.890325,0.803617,0.849020



=== B_rf_l1 only (check semantic feature usage) ===


,method,C,selected_total,selected_sem,roc_auc_mean,f1_macro_mean,accuracy_mean
5,B_rf_l1,0.2,11,0,0.897536,0.823395,0.864082
6,B_rf_l1,0.5,14,0,0.906942,0.815947,0.861052
7,B_rf_l1,1.0,20,1,0.902799,0.825560,0.863993
8,B_rf_l1,2.0,29,7,0.894637,0.809184,0.852050
9,B_rf_l1,5.0,37,14,0.890325,0.803617,0.849020


In [38]:
# Predict on no-answer test set and save submission
if 'final_model' not in globals():
    raise RuntimeError('final_model is not available. Run the training/evaluation cells first.')

test_proba = final_model.predict_proba(df_test)[:, 1]
test_pred = (test_proba >= 0.5).astype(int) + 1

submission = pd.DataFrame({
    'id': df_test['id'].values,
    'gender': test_pred
})
submission.to_csv('submission_feature_upgrade_v2.csv', index=False)

print('Saved: submission_feature_upgrade_v2.csv')
print(submission.head())

Saved: submission_feature_upgrade_v2.csv
   id  gender
0   1       1
1   2       1
2   3       2
3   4       1
4   5       2
